In [ ]:
# ============================================================
# NOTEBOOK 05 — MASTER SETUP
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
from pathlib import Path
import pandas as pd
import numpy as np
import nibabel as nib
import torch

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

BASE_PATH = Path(
    "/content/drive/MyDrive/Breast_Cancer_AI_Project"
)

DATA_PATH = BASE_PATH / "data" / "BreastDCEDL_ISPY1_min_crop"
CLEANED_PATH = BASE_PATH / "data" / "cleaned"
MASTER_CSV = CLEANED_PATH / "patient_master.csv"

NNUNET_RAW = BASE_PATH / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE_PATH / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE_PATH / "nnUNet_results"

NNUNET_DATASET = NNUNET_RAW / "Dataset001_BreastDCEDL"

# ------------------------------------------------------------
# 2. PERSISTENT INFERENCE DIRECTORIES
# ------------------------------------------------------------

INFERENCE_PATH = BASE_PATH / "inference"

# Validation
VAL_PATH = INFERENCE_PATH / "validation"
VAL_IMAGES = VAL_PATH / "images"
VAL_PROBABILITIES = VAL_PATH / "probabilities"

# Test
TEST_PATH = INFERENCE_PATH / "test"
TEST_IMAGES = TEST_PATH / "images"
TEST_PROBABILITIES = TEST_PATH / "probabilities"

# ------------------------------------------------------------
# 3. VISUALIZATION DIRECTORIES
# ------------------------------------------------------------

VISUALIZATION_PATH = BASE_PATH / "visualizations"

VAL_VIZ = VISUALIZATION_PATH / "validation"
TEST_VIZ = VISUALIZATION_PATH / "test"

# Create directories
for folder in [
    VAL_IMAGES,
    VAL_PROBABILITIES,
    TEST_IMAGES,
    TEST_PROBABILITIES,
    VAL_VIZ,
    TEST_VIZ
]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 4. nnU-Net ENVIRONMENT
# ------------------------------------------------------------

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

# ------------------------------------------------------------
# 5. LOAD PATIENT MASTER
# ------------------------------------------------------------

master = pd.read_csv(MASTER_CSV)

# ------------------------------------------------------------
# 6. BASIC CHECKS
# ------------------------------------------------------------

print("=" * 60)
print("PROJECT SETUP")
print("=" * 60)

print("\nPatients:", len(master))

print("\nSplit counts:")
print(master["split"].value_counts())

print("\nOriginal data exists:", DATA_PATH.exists())
print("nnU-Net dataset exists:", NNUNET_DATASET.exists())
print("nnU-Net results exists:", NNUNET_RESULTS.exists())

print("\nValidation images:", VAL_IMAGES)
print("Validation probabilities:", VAL_PROBABILITIES)

print("\nTest images:", TEST_IMAGES)
print("Test probabilities:", TEST_PROBABILITIES)

print("\nVisualization folders:")
print("Validation:", VAL_VIZ)
print("Test:", TEST_VIZ)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nnnUNetv2_predict:",
      shutil.which("nnUNetv2_predict"))

print("\n" + "=" * 60)
print("SETUP COMPLETE")
print("=" * 60)

In [ ]:
print("Validation image files:",
      len(list(VAL_IMAGES.glob("*.nii.gz"))))

print("Test image files:",
      len(list(TEST_IMAGES.glob("*.nii.gz"))))

print("Validation probability files:",
      len(list(VAL_PROBABILITIES.glob("*.npz"))))

print("Test probability files:",
      len(list(TEST_PROBABILITIES.glob("*.npz"))))

In [ ]:
import sys
import importlib.util

print("nnunetv2 installed:",
      importlib.util.find_spec("nnunetv2") is not None)

print("Python:", sys.version)

In [ ]:
!pip install -q nnunetv2

In [ ]:
import importlib.util
import shutil

print("nnunetv2 installed:",
      importlib.util.find_spec("nnunetv2") is not None)

print("nnUNetv2_predict:",
      shutil.which("nnUNetv2_predict"))

print("nnUNetv2_train:",
      shutil.which("nnUNetv2_train"))

In [ ]:
# ============================================================
# 5-FOLD ENSEMBLE INFERENCE — VALIDATION SET
# ============================================================

import subprocess

val_cmd = [
    "nnUNetv2_predict",
    "-d", "1",
    "-i", str(VAL_IMAGES),
    "-o", str(VAL_PROBABILITIES),
    "-f", "0", "1", "2", "3", "4",
    "-c", "3d_fullres",
    "-tr", "nnUNetTrainer",
    "-chk", "checkpoint_best.pth",
    "--save_probabilities",
    "-device", "cuda"
]

print("Starting 5-fold validation inference...")
print("Output:", VAL_PROBABILITIES)
print()

result = subprocess.run(val_cmd)

print()
print("=" * 60)
print("VALIDATION INFERENCE FINISHED")
print("Return code:", result.returncode)
print("=" * 60)

In [ ]:
# ============================================================
# VERIFY VALIDATION OUTPUTS
# ============================================================

val_npz = sorted(VAL_PROBABILITIES.glob("*.npz"))
val_nii = sorted(VAL_PROBABILITIES.glob("*.nii.gz"))

print("Validation probability maps:", len(val_npz))
print("Validation segmentation masks:", len(val_nii))

print("\nFirst 5 probability maps:")
for f in val_npz[:5]:
    print(" ", f.name)

print("\nFirst 5 segmentation masks:")
for f in val_nii[:5]:
    print(" ", f.name)

In [ ]:
# ============================================================
# 5-FOLD ENSEMBLE INFERENCE — TEST SET
# ============================================================

import subprocess

test_cmd = [
    "nnUNetv2_predict",
    "-d", "1",
    "-i", str(TEST_IMAGES),
    "-o", str(TEST_PROBABILITIES),
    "-f", "0", "1", "2", "3", "4",
    "-c", "3d_fullres",
    "-tr", "nnUNetTrainer",
    "-chk", "checkpoint_best.pth",
    "--save_probabilities",
    "-device", "cuda"
]

print("Starting 5-fold test inference...")
print("Output:", TEST_PROBABILITIES)
print()

result = subprocess.run(test_cmd)

print()
print("=" * 60)
print("TEST INFERENCE FINISHED")
print("Return code:", result.returncode)
print("=" * 60)

In [ ]:
# ============================================================
# VERIFY TEST OUTPUTS
# ============================================================

test_npz = sorted(TEST_PROBABILITIES.glob("*.npz"))
test_nii = sorted(TEST_PROBABILITIES.glob("*.nii.gz"))

print("Test probability maps:", len(test_npz))
print("Test segmentation masks:", len(test_nii))

print("\nFirst 5 probability maps:")
for f in test_npz[:5]:
    print(" ", f.name)

print("\nFirst 5 segmentation masks:")
for f in test_nii[:5]:
    print(" ", f.name)

In [ ]:
# ============================================================
# 3D VISUALIZATION SETUP
# ============================================================

import numpy as np
import nibabel as nib
import pandas as pd
from pathlib import Path

from skimage.measure import marching_cubes

import plotly.graph_objects as go
from IPython.display import display

print("3D visualization libraries loaded.")

In [ ]:
# ============================================================
# FIRST PATIENT
# ============================================================

PATIENT_ID = "ISPY1_1134"

# Find patient information
patient_row = master[master["pid"] == PATIENT_ID].iloc[0]

print("Patient:", PATIENT_ID)
print("Split:", patient_row["split"])
print("pCR:", patient_row["pCR"])

print("\nGround-truth mask:")
print(patient_row["mask"])

print("\nProbability map:")
print(TEST_PROBABILITIES / f"{PATIENT_ID}.npz")

In [ ]:
# ============================================================
# LOAD PREDICTION + GROUND TRUTH
# ============================================================

prob_file = TEST_PROBABILITIES / f"{PATIENT_ID}.npz"

# Ground-truth mask path stored in patient_master.csv
gt_file = Path(patient_row["mask"])

# Load probability map
prob_data = np.load(prob_file)

print("Probability keys:", prob_data.files)

probability = prob_data["probabilities"]

print("\nOriginal probability shape:", probability.shape)
print("Probability dtype:", probability.dtype)

# Class 1 = tumor
tumor_probability = probability[1]

# IMPORTANT:
# nnU-Net probability array uses a different axis ordering
# from the exported segmentation / ground-truth mask.
#
# We previously verified that:
#
# np.transpose(probabilities[1], (2,1,0))
#
# exactly matches the nnU-Net segmentation orientation.

tumor_probability = np.transpose(
    tumor_probability,
    (2, 1, 0)
)

# Load ground truth
gt_img = nib.load(gt_file)
ground_truth = gt_img.get_fdata()

# Convert to binary
ground_truth = ground_truth > 0

print("\nAfter orientation correction:")
print("Tumor probability shape:", tumor_probability.shape)
print("Ground truth shape:", ground_truth.shape)

print("\nProbability range:",
      tumor_probability.min(),
      "to",
      tumor_probability.max())

print("GT tumor voxels:", ground_truth.sum())

In [ ]:
# ============================================================
# FINAL PREDICTION — THRESHOLD 0.65
# ============================================================

THRESHOLD = 0.65

predicted_tumor = tumor_probability >= THRESHOLD

print("Threshold:", THRESHOLD)
print("Predicted tumor voxels:", predicted_tumor.sum())
print("Ground-truth tumor voxels:", ground_truth.sum())

In [ ]:
intersection = np.logical_and(
    predicted_tumor,
    ground_truth
).sum()

dice = (
    2 * intersection /
    (predicted_tumor.sum() + ground_truth.sum())
)

print("Dice:", dice)

In [ ]:
# ============================================================
# 3D WEB VIEWER — DEPENDENCIES
# ============================================================

!pip install -q plotly scikit-image

In [ ]:
# ============================================================
# FIXED INTERACTIVE 3D TUMOR VIEWER
# ============================================================

import json

# ------------------------------------------------------------
# Convert patient data directly to JavaScript JSON
# ------------------------------------------------------------

patient_json = json.dumps(
    all_patients,
    separators=(",", ":")
)

# IMPORTANT:
# Do NOT json.dumps(patient_json) again.
# patient_json is already valid JSON.

patient_data_js = patient_json

# ------------------------------------------------------------
# HTML
# ------------------------------------------------------------

html = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>Breast Cancer AI — 3D Tumor Viewer</title>

<style>

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    background: #0b0f14;
    color: #e6edf3;
    font-family: Arial, Helvetica, sans-serif;
}}

.header {{
    padding: 24px 32px 18px;
    border-bottom: 1px solid #202832;
    background: #0f141b;
}}

.title {{
    font-size: 26px;
    font-weight: 700;
}}

.subtitle {{
    margin-top: 6px;
    color: #8b98a8;
    font-size: 14px;
}}

.controls {{
    display: flex;
    gap: 24px;
    align-items: flex-end;
    flex-wrap: wrap;
    padding: 18px 32px;
    background: #0f141b;
    border-bottom: 1px solid #202832;
}}

.control-group {{
    display: flex;
    flex-direction: column;
    gap: 7px;
}}

label {{
    font-size: 12px;
    color: #8b98a8;
}}

select {{
    background: #151b23;
    color: #e6edf3;
    border: 1px solid #303a46;
    border-radius: 6px;
    padding: 10px 35px 10px 12px;
    font-size: 14px;
    min-width: 250px;
}}

.checkbox-row {{
    display: flex;
    gap: 18px;
    align-items: center;
    padding-bottom: 4px;
}}

.checkbox-row label {{
    display: flex;
    flex-direction: row;
    gap: 7px;
    align-items: center;
    color: #c9d1d9;
    font-size: 13px;
}}

.main {{
    display: grid;
    grid-template-columns: minmax(0, 1fr) 310px;
    min-height: calc(100vh - 150px);
}}

.viewer {{
    min-height: 720px;
    background: #080c11;
}}

.sidebar {{
    border-left: 1px solid #202832;
    background: #0f141b;
    padding: 24px 20px;
    overflow-y: auto;
}}

.patient-name {{
    font-size: 21px;
    font-weight: 700;
    margin-bottom: 6px;
}}

.badge {{
    display: inline-block;
    padding: 4px 9px;
    border-radius: 12px;
    background: #202a35;
    color: #9fb1c3;
    font-size: 11px;
    margin-bottom: 22px;
}}

.section-title {{
    color: #8b98a8;
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 1px;
    margin-top: 20px;
    margin-bottom: 10px;
}}

.metric {{
    display: flex;
    justify-content: space-between;
    padding: 9px 0;
    border-bottom: 1px solid #1d252f;
}}

.metric-name {{
    color: #9aa7b5;
}}

.metric-value {{
    font-weight: 600;
}}

.note {{
    margin-top: 22px;
    padding: 12px;
    background: #151b23;
    border-radius: 7px;
    color: #8b98a8;
    font-size: 12px;
    line-height: 1.5;
}}

@media (max-width: 900px) {{

    .main {{
        grid-template-columns: 1fr;
    }}

    .sidebar {{
        border-left: none;
        border-top: 1px solid #202832;
    }}

}}

</style>

</head>


<body>


<div class="header">

    <div class="title">
        Breast Cancer AI — 3D Tumor Viewer
    </div>

    <div class="subtitle">
        Interactive nnU-Net tumor segmentation
        · Validation-selected threshold = 0.65
    </div>

</div>


<div class="controls">

    <div class="control-group">

        <label>PATIENT</label>

        <select id="patientSelect"></select>

    </div>


    <div class="checkbox-row">

        <label>
            <input type="checkbox" id="showGT" checked>
            Ground Truth
        </label>

        <label>
            <input type="checkbox" id="showPred" checked>
            Prediction
        </label>

    </div>

</div>


<div class="main">


    <div id="viewer" class="viewer"></div>


    <div class="sidebar">

        <div id="patientName" class="patient-name"></div>

        <div id="patientSplit" class="badge"></div>


        <div class="section-title">
            Response
        </div>

        <div class="metric">

            <span class="metric-name">
                pCR
            </span>

            <span id="pcr" class="metric-value"></span>

        </div>


        <div class="section-title">
            Segmentation Performance
        </div>


        <div class="metric">
            <span class="metric-name">Dice</span>
            <span id="dice" class="metric-value"></span>
        </div>


        <div class="metric">
            <span class="metric-name">IoU</span>
            <span id="iou" class="metric-value"></span>
        </div>


        <div class="metric">
            <span class="metric-name">Precision</span>
            <span id="precision" class="metric-value"></span>
        </div>


        <div class="metric">
            <span class="metric-name">Recall</span>
            <span id="recall" class="metric-value"></span>
        </div>


        <div class="section-title">
            Tumor Volume
        </div>


        <div class="metric">
            <span class="metric-name">Ground Truth</span>
            <span id="gtVoxels" class="metric-value"></span>
        </div>


        <div class="metric">
            <span class="metric-name">Prediction</span>
            <span id="predVoxels" class="metric-value"></span>
        </div>


        <div class="metric">
            <span class="metric-name">Threshold</span>
            <span class="metric-value">0.65</span>
        </div>


        <div class="section-title">
            Dataset
        </div>


        <div class="metric">
            <span class="metric-name">Split</span>
            <span id="datasetSplit" class="metric-value"></span>
        </div>


        <div class="metric">
            <span class="metric-name">Image Shape</span>
            <span id="imageShape" class="metric-value"></span>
        </div>


        <div class="note">

            Drag to rotate the tumor.<br>
            Scroll to zoom.<br>
            Right-click + drag to pan.<br><br>

            Blue = Ground Truth<br>
            Red = nnU-Net Prediction

        </div>

    </div>

</div>


<script>

/* ============================================================
   PATIENT DATA
   ============================================================ */

const patientData = {patient_data_js};

console.log(
    "Loaded patients:",
    patientData.length
);


/* ============================================================
   PATIENT DROPDOWN
   ============================================================ */

const patientSelect =
    document.getElementById("patientSelect");


patientData.forEach((patient, index) => {{

    const option =
        document.createElement("option");

    option.value = index;

    option.textContent =
        patient.pid +
        " — " +
        patient.split;

    patientSelect.appendChild(option);

}});


/* ============================================================
   METRICS
   ============================================================ */

function formatMetric(value) {{

    return Number(value).toFixed(4);

}}


/* ============================================================
   SIDEBAR
   ============================================================ */

function updateSidebar(patient) {{

    document.getElementById(
        "patientName"
    ).textContent = patient.pid;


    document.getElementById(
        "patientSplit"
    ).textContent =
        patient.split.toUpperCase();


    document.getElementById(
        "dice"
    ).textContent =
        formatMetric(patient.dice);


    document.getElementById(
        "iou"
    ).textContent =
        formatMetric(patient.iou);


    document.getElementById(
        "precision"
    ).textContent =
        formatMetric(patient.precision);


    document.getElementById(
        "recall"
    ).textContent =
        formatMetric(patient.recall);


    document.getElementById(
        "gtVoxels"
    ).textContent =
        patient.gt_voxels.toLocaleString();


    document.getElementById(
        "predVoxels"
    ).textContent =
        patient.pred_voxels.toLocaleString();


    document.getElementById(
        "pcr"
    ).textContent =
        patient.pcr === null
        ? "N/A"
        : patient.pcr === 1
            ? "Yes"
            : "No";


    document.getElementById(
        "datasetSplit"
    ).textContent =
        patient.split;


    if (patient.gt_vertices.length > 0) {{

        const zValues =
            patient.gt_vertices.map(
                v => v[2]
            );

        const yValues =
            patient.gt_vertices.map(
                v => v[1]
            );

        const xValues =
            patient.gt_vertices.map(
                v => v[0]
            );

        document.getElementById(
            "imageShape"
        ).textContent =
            "3D tumor mesh";

    }} else {{

        document.getElementById(
            "imageShape"
        ).textContent =
            "N/A";

    }}

}}


/* ============================================================
   MESH
   ============================================================ */

function makeMesh(
    vertices,
    faces,
    name,
    color,
    opacity
) {{

    if (
        !vertices ||
        vertices.length === 0
    ) {{
        return null;
    }}


    return {{

        type: "mesh3d",

        x: vertices.map(v => v[0]),
        y: vertices.map(v => v[1]),
        z: vertices.map(v => v[2]),

        i: faces.map(f => f[0]),
        j: faces.map(f => f[1]),
        k: faces.map(f => f[2]),

        name: name,

        color: color,

        opacity: opacity,

        flatshading: false,

        hoverinfo: "skip"

    }};

}}


/* ============================================================
   RENDER PATIENT
   ============================================================ */

function renderPatient(index) {{

    const patient =
        patientData[index];

    updateSidebar(patient);


    const traces = [];


    const showGT =
        document.getElementById(
            "showGT"
        ).checked;


    const showPred =
        document.getElementById(
            "showPred"
        ).checked;


    if (showGT) {{

        const gtTrace =
            makeMesh(
                patient.gt_vertices,
                patient.gt_faces,
                "Ground Truth",
                "#3b82f6",
                0.48
            );

        if (gtTrace !== null) {{
            traces.push(gtTrace);
        }}

    }}


    if (showPred) {{

        const predTrace =
            makeMesh(
                patient.pred_vertices,
                patient.pred_faces,
                "Prediction",
                "#ef4444",
                0.58
            );

        if (predTrace !== null) {{
            traces.push(predTrace);
        }}

    }}


    const layout = {{

        paper_bgcolor: "#080c11",

        plot_bgcolor: "#080c11",

        font: {{
            color: "#d6dee7"
        }},

        margin: {{
            l: 0,
            r: 0,
            t: 20,
            b: 0
        }},

        scene: {{

            bgcolor: "#080c11",

            xaxis: {{
                title: "",
                showgrid: false,
                zeroline: false,
                showticklabels: false
            }},

            yaxis: {{
                title: "",
                showgrid: false,
                zeroline: false,
                showticklabels: false
            }},

            zaxis: {{
                title: "",
                showgrid: false,
                zeroline: false,
                showticklabels: false
            }},

            aspectmode: "data"

        }},

        legend: {{

            bgcolor:
                "rgba(15,20,27,0.9)",

            bordercolor:
                "#303a46",

            borderwidth: 1

        }}

    }};


    Plotly.react(
        "viewer",
        traces,
        layout,
        {{
            responsive: true,
            displaylogo: false
        }}
    );

}}


/* ============================================================
   EVENTS
   ============================================================ */

patientSelect.addEventListener(
    "change",
    () => {{
        renderPatient(
            Number(patientSelect.value)
        );
    }}
);


document.getElementById(
    "showGT"
).addEventListener(
    "change",
    () => {{
        renderPatient(
            Number(patientSelect.value)
        );
    }}
);


document.getElementById(
    "showPred"
).addEventListener(
    "change",
    () => {{
        renderPatient(
            Number(patientSelect.value)
        );
    }}
);


/* ============================================================
   INITIAL PATIENT
   ============================================================ */

if (patientData.length > 0) {{

    patientSelect.value = "0";

    renderPatient(0);

}}

</script>


<script>
{plotly_js}
</script>


</body>

</html>
"""


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT_HTML.write_text(
    html,
    encoding="utf-8"
)

print("=" * 70)
print("FIXED 3D VIEWER CREATED")
print("=" * 70)

print("\nPatients:", len(all_patients))
print("Validation:", sum(
    p["split"] == "validation"
    for p in all_patients
))
print("Test:", sum(
    p["split"] == "test"
    for p in all_patients
))

print("\nSaved to:")
print(OUTPUT_HTML)

print(
    "\nFile size:",
    f"{OUTPUT_HTML.stat().st_size / (1024**2):.2f} MB"
)

In [ ]:
# ============================================================
# BREAST CANCER AI — DETAILED INTERACTIVE 3D TUMOR VIEWER
# ============================================================

import json
import numpy as np
import nibabel as nib
import pandas as pd

from pathlib import Path
from skimage.measure import marching_cubes
from nibabel.affines import apply_affine

from plotly.offline import get_plotlyjs


# ============================================================
# SETTINGS
# ============================================================

THRESHOLD = 0.65

OUTPUT_HTML = (
    VISUALIZATION_PATH /
    "Breast_Cancer_3D_Tumor_Viewer.html"
)

MARCHING_CUBES_STEP = 2


# ============================================================
# METRICS
# ============================================================

def calculate_detailed_metrics(pred, gt):

    pred = pred.astype(bool)
    gt = gt.astype(bool)

    tp = np.logical_and(
        pred,
        gt
    ).sum()

    fp = np.logical_and(
        pred,
        ~gt
    ).sum()

    fn = np.logical_and(
        ~pred,
        gt
    ).sum()

    tn = np.logical_and(
        ~pred,
        ~gt
    ).sum()

    dice_den = (
        pred.sum() +
        gt.sum()
    )

    iou_den = (
        tp + fp + fn
    )

    dice = (
        2 * tp / dice_den
        if dice_den > 0
        else 1.0
    )

    iou = (
        tp / iou_den
        if iou_den > 0
        else 1.0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
        if (tp + tn + fp + fn) > 0
        else 0.0
    )

    return {

        "tp": int(tp),

        "fp": int(fp),

        "fn": int(fn),

        "tn": int(tn),

        "dice": float(dice),

        "iou": float(iou),

        "precision": float(precision),

        "recall": float(recall),

        "specificity": float(specificity),

        "accuracy": float(accuracy),

        "gt_voxels": int(gt.sum()),

        "pred_voxels": int(pred.sum()),

        "overlap_voxels": int(tp),

        "gt_only_voxels": int(
            np.logical_and(
                gt,
                ~pred
            ).sum()
        ),

        "pred_only_voxels": int(
            np.logical_and(
                pred,
                ~gt
            ).sum()
        )
    }


# ============================================================
# SAFE MASK → MESH
# ============================================================

def mask_to_mesh(
    mask,
    affine,
    step_size=2
):

    mask = mask.astype(np.uint8)

    # --------------------------------------------------------
    # IMPORTANT:
    # Empty masks have no surface.
    # Return None instead of crashing.
    # --------------------------------------------------------

    if mask.sum() == 0:

        return None


    # --------------------------------------------------------
    # If only one voxel exists, marching cubes can also fail.
    # Padding gives us a proper boundary.
    # --------------------------------------------------------

    padded = np.pad(
        mask,
        1,
        mode="constant"
    )


    try:

        verts, faces, normals, values = (
            marching_cubes(
                padded,
                level=0.5,
                step_size=step_size
            )
        )

    except RuntimeError:

        return None


    # Remove padding offset

    verts -= 1


    # Convert voxel coordinates
    # to physical coordinates

    verts_world = apply_affine(
        affine,
        verts
    )


    return (
        verts_world,
        faces
    )


# ============================================================
# MESH → JSON
# ============================================================

def mesh_to_dict(mesh):

    if mesh is None:

        return {
            "vertices": [],
            "faces": []
        }


    vertices, faces = mesh


    return {

        "vertices":
            vertices.tolist(),

        "faces":
            faces.tolist()

    }


# ============================================================
# PATIENTS
# ============================================================

validation_rows = master[
    master["split"] == "validation"
].copy()

test_rows = master[
    master["split"] == "test"
].copy()

patient_rows = pd.concat(
    [
        validation_rows,
        test_rows
    ],
    ignore_index=True
)


# ============================================================
# BUILD DATA
# ============================================================

all_patients = []


print("=" * 70)
print("BUILDING DETAILED 3D TUMOR VIEWER")
print("=" * 70)

print(
    f"Patients: {len(patient_rows)} "
    f"(Validation={len(validation_rows)}, "
    f"Test={len(test_rows)})"
)


for idx, row in patient_rows.iterrows():

    pid = row["pid"]

    split = row["split"]


    print(
        f"[{idx + 1:02d}/{len(patient_rows)}] "
        f"{pid} ({split})"
    )


    # ========================================================
    # PROBABILITY FILE
    # ========================================================

    if split == "validation":

        probability_file = (
            VAL_PROBABILITIES /
            f"{pid}.npz"
        )

    else:

        probability_file = (
            TEST_PROBABILITIES /
            f"{pid}.npz"
        )


    # ========================================================
    # LOAD PROBABILITY
    # ========================================================

    probability_data = np.load(
        probability_file
    )

    tumor_probability = (
        probability_data[
            "probabilities"
        ][1]
    )


    # Established orientation correction

    tumor_probability = np.transpose(
        tumor_probability,
        (2, 1, 0)
    )


    # ========================================================
    # PREDICTION
    # ========================================================

    prediction = (
        tumor_probability >= THRESHOLD
    )


    # ========================================================
    # GROUND TRUTH
    # ========================================================

    gt_file = Path(
        row["mask"]
    )

    gt_img = nib.load(
        gt_file
    )

    ground_truth = (
        gt_img.get_fdata() > 0
    )


    # ========================================================
    # METRICS
    # ========================================================

    metrics = calculate_detailed_metrics(
        prediction,
        ground_truth
    )


    # ========================================================
    # REGIONS
    # ========================================================

    gt_only = np.logical_and(
        ground_truth,
        ~prediction
    )

    overlap = np.logical_and(
        ground_truth,
        prediction
    )

    pred_only = np.logical_and(
        prediction,
        ~ground_truth
    )


    # ========================================================
    # MESHES
    # ========================================================

    gt_only_mesh = mask_to_mesh(
        gt_only,
        gt_img.affine,
        MARCHING_CUBES_STEP
    )

    overlap_mesh = mask_to_mesh(
        overlap,
        gt_img.affine,
        MARCHING_CUBES_STEP
    )

    pred_only_mesh = mask_to_mesh(
        pred_only,
        gt_img.affine,
        MARCHING_CUBES_STEP
    )


    # ========================================================
    # VOXEL VOLUME
    # ========================================================

    voxel_volume_mm3 = abs(
        np.linalg.det(
            gt_img.affine[:3, :3]
        )
    )

    voxel_volume_cm3 = (
        voxel_volume_mm3 / 1000.0
    )


    gt_volume_cm3 = (
        metrics["gt_voxels"] *
        voxel_volume_cm3
    )

    pred_volume_cm3 = (
        metrics["pred_voxels"] *
        voxel_volume_cm3
    )

    overlap_volume_cm3 = (
        metrics["overlap_voxels"] *
        voxel_volume_cm3
    )

    gt_only_volume_cm3 = (
        metrics["gt_only_voxels"] *
        voxel_volume_cm3
    )

    pred_only_volume_cm3 = (
        metrics["pred_only_voxels"] *
        voxel_volume_cm3
    )


    # ========================================================
    # VOLUME DIFFERENCE
    # ========================================================

    volume_difference_cm3 = (
        pred_volume_cm3 -
        gt_volume_cm3
    )

    volume_difference_percent = (

        (
            pred_volume_cm3 -
            gt_volume_cm3
        )
        / gt_volume_cm3
        * 100

        if gt_volume_cm3 > 0

        else 0
    )


    # ========================================================
    # OVERLAP PERCENTAGE
    # ========================================================

    overlap_percentage_gt = (

        metrics["overlap_voxels"]
        / metrics["gt_voxels"]
        * 100

        if metrics["gt_voxels"] > 0

        else 0
    )


    overlap_percentage_pred = (

        metrics["overlap_voxels"]
        / metrics["pred_voxels"]
        * 100

        if metrics["pred_voxels"] > 0

        else 0
    )


    # ========================================================
    # STORE
    # ========================================================

    patient_data = {

        "pid":
            pid,

        "split":
            split,

        "pcr":
            (
                None
                if pd.isna(row["pCR"])
                else float(row["pCR"])
            ),

        "dice":
            metrics["dice"],

        "iou":
            metrics["iou"],

        "precision":
            metrics["precision"],

        "recall":
            metrics["recall"],

        "specificity":
            metrics["specificity"],

        "accuracy":
            metrics["accuracy"],

        "tp":
            metrics["tp"],

        "fp":
            metrics["fp"],

        "fn":
            metrics["fn"],

        "tn":
            metrics["tn"],

        "gt_voxels":
            metrics["gt_voxels"],

        "pred_voxels":
            metrics["pred_voxels"],

        "overlap_voxels":
            metrics["overlap_voxels"],

        "gt_only_voxels":
            metrics["gt_only_voxels"],

        "pred_only_voxels":
            metrics["pred_only_voxels"],

        "gt_volume_cm3":
            float(gt_volume_cm3),

        "pred_volume_cm3":
            float(pred_volume_cm3),

        "overlap_volume_cm3":
            float(overlap_volume_cm3),

        "gt_only_volume_cm3":
            float(gt_only_volume_cm3),

        "pred_only_volume_cm3":
            float(pred_only_volume_cm3),

        "volume_difference_cm3":
            float(volume_difference_cm3),

        "volume_difference_percent":
            float(volume_difference_percent),

        "overlap_percentage_gt":
            float(overlap_percentage_gt),

        "overlap_percentage_pred":
            float(overlap_percentage_pred),

        "image_shape":
            list(
                ground_truth.shape
            ),

        "voxel_spacing":
            [
                float(x)
                for x in gt_img.header.get_zooms()[:3]
            ],

        "gt_only":
            mesh_to_dict(
                gt_only_mesh
            ),

        "overlap":
            mesh_to_dict(
                overlap_mesh
            ),

        "pred_only":
            mesh_to_dict(
                pred_only_mesh
            )
    }


    all_patients.append(
        patient_data
    )


# ============================================================
# SERIALIZE
# ============================================================

patient_json = json.dumps(
    all_patients,
    separators=(",", ":")
)


print("\nAll patients processed.")

print(
    "Patient data size:",
    f"{len(patient_json)/(1024**2):.2f} MB"
)


# ============================================================
# PLOTLY
# ============================================================

plotly_js = get_plotlyjs()


# ============================================================
# HTML
# ============================================================

html = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>
Breast Cancer AI — 3D Tumor Viewer
</title>


<style>

/* ==========================================================
   GLOBAL
   ========================================================== */

* {{
    box-sizing: border-box;
}}

html,
body {{
    margin: 0;
    padding: 0;
    width: 100%;
    min-height: 100%;
}}

body {{
    background: #080c11;
    color: #e6edf3;
    font-family:
        Arial,
        Helvetica,
        sans-serif;
}}


/* ==========================================================
   HEADER
   ========================================================== */

.header {{
    padding: 22px 32px 19px;
    background: #0d131b;
    border-bottom: 1px solid #202a35;
}}

.header-row {{
    display: flex;
    justify-content: space-between;
    align-items: center;
}}

.brand {{
    display: flex;
    align-items: center;
    gap: 14px;
}}

.ribbon {{
    font-size: 34px;
}}

.title {{
    font-size: 27px;
    font-weight: 700;
    letter-spacing: 0.2px;
}}

.subtitle {{
    margin-top: 6px;
    color: #8fa5bd;
    font-size: 14px;
}}

.header-right {{
    text-align: right;
    color: #8195ab;
    font-size: 10px;
    line-height: 1.7;
    letter-spacing: 1.6px;
}}


/* ==========================================================
   CONTROLS
   ========================================================== */

.controls {{
    display: flex;
    align-items: flex-end;
    gap: 22px;
    flex-wrap: wrap;
    padding: 17px 32px;
    background: #0d131b;
    border-bottom: 1px solid #202a35;
}}

.control-group {{
    display: flex;
    flex-direction: column;
    gap: 6px;
}}

.control-label {{
    font-size: 10px;
    color: #8499b0;
    letter-spacing: 1px;
}}

select {{
    min-width: 215px;
    padding: 10px 35px 10px 12px;
    border: 1px solid #303d4c;
    border-radius: 7px;
    background: #151d27;
    color: #e6edf3;
    font-size: 13px;
    outline: none;
}}

select:focus {{
    border-color: #5d7fa5;
}}


/* ==========================================================
   CHECKBOXES
   ========================================================== */

.checkbox-row {{
    display: flex;
    gap: 17px;
    align-items: center;
    padding-bottom: 3px;
}}

.checkbox-label {{
    display: flex;
    align-items: center;
    gap: 7px;
    color: #cbd5df;
    font-size: 13px;
    cursor: pointer;
}}

.checkbox-label input {{
    width: 15px;
    height: 15px;
}}


/* ==========================================================
   MAIN
   ========================================================== */

.main {{
    display: grid;
    grid-template-columns:
        minmax(0, 1fr)
        360px;

    height:
        calc(100vh - 145px);

    min-height:
        680px;
}}


/* ==========================================================
   VIEWER
   ========================================================== */

.viewer-wrapper {{
    position: relative;
    padding: 17px;
    background: #080c11;
}}

.viewer-card {{
    width: 100%;
    height: 100%;
    border: 1px solid #202d3b;
    border-radius: 10px;

    background:
        radial-gradient(
            circle at center,
            #101925 0%,
            #080c11 68%
        );

    overflow: hidden;
}}

#viewer {{
    width: 100%;
    height: 100%;
}}


/* ==========================================================
   LEGEND
   ========================================================== */

.legend {{
    position: absolute;
    top: 31px;
    left: 31px;

    padding: 11px 13px;

    border: 1px solid #263342;
    border-radius: 8px;

    background:
        rgba(8,13,20,0.90);

    backdrop-filter:
        blur(5px);
}}

.legend-item {{
    display: flex;
    align-items: center;
    gap: 8px;
    margin: 5px 0;

    font-size: 11px;
    color: #d5dee7;
}}

.dot {{
    width: 10px;
    height: 10px;
    border-radius: 50%;
}}

.dot-blue {{
    background: #3b82f6;
}}

.dot-purple {{
    background: #a855f7;
}}

.dot-red {{
    background: #ef4444;
}}


/* ==========================================================
   VIEWER FOOTER
   ========================================================== */

.viewer-footer {{
    position: absolute;

    bottom: 29px;
    left: 31px;
    right: 31px;

    display: flex;
    justify-content: space-between;
    align-items: center;

    padding: 10px 13px;

    border:
        1px solid #263342;

    border-radius:
        8px;

    background:
        rgba(8,13,20,0.90);

    color:
        #8295aa;

    font-size:
        10px;
}}

.footer-controls {{
    display:
        flex;

    gap:
        18px;
}}


/* ==========================================================
   SIDEBAR
   ========================================================== */

.sidebar {{
    border-left:
        1px solid #202a35;

    background:
        #0d131b;

    padding:
        23px 21px;

    overflow-y:
        auto;
}}


/* ==========================================================
   PATIENT TITLE
   ========================================================== */

.patient-title-row {{
    display:
        flex;

    align-items:
        center;

    gap:
        9px;
}}

.patient-name {{
    font-size:
        21px;

    font-weight:
        700;
}}

.badge {{
    padding:
        4px 9px;

    border-radius:
        12px;

    background:
        #202b3b;

    color:
        #abc0d6;

    font-size:
        9px;

    letter-spacing:
        0.8px;
}}


/* ==========================================================
   PCR
   ========================================================== */

.pcr-row {{
    display:
        flex;

    justify-content:
        space-between;

    padding:
        13px 0;

    margin-top:
        15px;

    border-top:
        1px solid #222d38;

    border-bottom:
        1px solid #222d38;
}}

.pcr-label {{
    color:
        #9eacba;

    font-size:
        13px;
}}

.pcr-value {{
    font-weight:
        700;
}}


/* ==========================================================
   SECTION
   ========================================================== */

.section-title {{
    margin-top:
        20px;

    margin-bottom:
        7px;

    color:
        #87a2bf;

    font-size:
        10px;

    letter-spacing:
        1.4px;

    text-transform:
        uppercase;
}}


/* ==========================================================
   METRICS
   ========================================================== */

.metric {{
    display:
        flex;

    justify-content:
        space-between;

    align-items:
        center;

    padding:
        7px 0;

    border-bottom:
        1px solid #1c252e;
}}

.metric-name {{
    color:
        #9baab9;

    font-size:
        12px;
}}

.metric-value {{
    color:
        #e5edf4;

    font-size:
        12px;

    font-weight:
        600;
}}


/* ==========================================================
   INFO BOX
   ========================================================== */

.info-box {{
    margin-top:
        20px;

    padding:
        12px;

    border:
        1px solid #263549;

    border-radius:
        8px;

    background:
        #111a26;

    color:
        #91a3b6;

    font-size:
        10px;

    line-height:
        1.55;
}}


/* ==========================================================
   RESPONSIVE
   ========================================================== */

@media (max-width: 950px) {{

    .header-right {{
        display: none;
    }}

    .main {{
        grid-template-columns:
            1fr;

        height:
            auto;
    }}

    .viewer-wrapper {{
        height:
            70vh;
    }}

    .sidebar {{
        border-left:
            none;

        border-top:
            1px solid #202a35;
    }}

}}

</style>

</head>


<body>


<!-- ========================================================
     HEADER
     ======================================================== -->

<div class="header">

    <div class="header-row">

        <div class="brand">

            <div class="ribbon">
                🎗
            </div>

            <div>

                <div class="title">
                    Breast Cancer AI — 3D Tumor Viewer
                </div>

                <div class="subtitle">
                    Interactive nnU-Net tumor segmentation
                    · Validation-selected threshold = 0.65
                </div>

            </div>

        </div>


        <div class="header-right">

            I-SPY1 BREAST MRI ANALYSIS<br>

            AI FOR TUMOR SEGMENTATION<br>

            5-FOLD nnU-NET ENSEMBLE

        </div>

    </div>

</div>


<!-- ========================================================
     CONTROLS
     ======================================================== -->

<div class="controls">


    <div class="control-group">

        <div class="control-label">
            DATASET
        </div>

        <select id="datasetSelect">

            <option value="all">
                All Patients
            </option>

            <option value="validation">
                Validation
            </option>

            <option value="test">
                Test
            </option>

        </select>

    </div>


    <div class="control-group">

        <div class="control-label">
            PATIENT
        </div>

        <select id="patientSelect">
        </select>

    </div>


    <div class="checkbox-row">


        <label class="checkbox-label">

            <input
                type="checkbox"
                id="showGT"
                checked
            >

            GT only

        </label>


        <label class="checkbox-label">

            <input
                type="checkbox"
                id="showOverlap"
                checked
            >

            Overlap

        </label>


        <label class="checkbox-label">

            <input
                type="checkbox"
                id="showPred"
                checked
            >

            Prediction only

        </label>


    </div>

</div>


<!-- ========================================================
     MAIN
     ======================================================== -->

<div class="main">


    <div class="viewer-wrapper">


        <div class="viewer-card">

            <div id="viewer"></div>

        </div>


        <div class="legend">


            <div class="legend-item">

                <span class="dot dot-blue"></span>

                GT only

            </div>


            <div class="legend-item">

                <span class="dot dot-purple"></span>

                Correct overlap

            </div>


            <div class="legend-item">

                <span class="dot dot-red"></span>

                Prediction only

            </div>


        </div>


        <div class="viewer-footer">

            <div class="footer-controls">

                <span>
                    🖱 Drag — Rotate
                </span>

                <span>
                    Scroll — Zoom
                </span>

                <span>
                    Right drag — Pan
                </span>

            </div>


            <span>
                3D surface reconstruction
            </span>

        </div>


    </div>


    <!-- ====================================================
         SIDEBAR
         ==================================================== -->

    <div class="sidebar">


        <div class="patient-title-row">

            <div
                id="patientName"
                class="patient-name"
            >
            </div>

            <div
                id="patientBadge"
                class="badge"
            >
            </div>

        </div>


        <!-- PCR -->

        <div class="pcr-row">

            <span class="pcr-label">
                pCR
            </span>

            <span
                id="pcr"
                class="pcr-value"
            >
            </span>

        </div>


        <!-- SEGMENTATION -->

        <div class="section-title">
            Segmentation Performance
        </div>


        <div class="metric">

            <span class="metric-name">
                Dice
            </span>

            <span
                id="dice"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                IoU
            </span>

            <span
                id="iou"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Precision
            </span>

            <span
                id="precision"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Recall
            </span>

            <span
                id="recall"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Specificity
            </span>

            <span
                id="specificity"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Accuracy
            </span>

            <span
                id="accuracy"
                class="metric-value"
            >
            </span>

        </div>


        <!-- CONFUSION -->

        <div class="section-title">
            Segmentation Components
        </div>


        <div class="metric">

            <span class="metric-name">
                True Positive
            </span>

            <span
                id="tp"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                False Positive
            </span>

            <span
                id="fp"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                False Negative
            </span>

            <span
                id="fn"
                class="metric-value"
            >
            </span>

        </div>


        <!-- VOLUME -->

        <div class="section-title">
            Tumor Volume
        </div>


        <div class="metric">

            <span class="metric-name">
                Ground Truth
            </span>

            <span
                id="gtVolume"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Prediction
            </span>

            <span
                id="predVolume"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Difference
            </span>

            <span
                id="volumeDifference"
                class="metric-value"
            >
            </span>

        </div>


        <!-- OVERLAP -->

        <div class="section-title">
            Spatial Agreement
        </div>


        <div class="metric">

            <span class="metric-name">
                Correct overlap
            </span>

            <span
                id="overlapVolume"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                GT overlap
            </span>

            <span
                id="overlapGT"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Prediction overlap
            </span>

            <span
                id="overlapPred"
                class="metric-value"
            >
            </span>

        </div>


        <!-- DATASET -->

        <div class="section-title">
            Dataset Information
        </div>


        <div class="metric">

            <span class="metric-name">
                Split
            </span>

            <span
                id="split"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Image shape
            </span>

            <span
                id="imageShape"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Voxel spacing
            </span>

            <span
                id="voxelSpacing"
                class="metric-value"
            >
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Threshold
            </span>

            <span class="metric-value">
                0.65
            </span>

        </div>


        <div class="metric">

            <span class="metric-name">
                Model
            </span>

            <span class="metric-value">
                5-fold nnU-Net
            </span>

        </div>


        <!-- INFO -->

        <div class="info-box">

            <strong>
                How to interpret the 3D comparison
            </strong>

            <br><br>

            <b>Blue:</b>
            tumor present in the ground-truth
            mask but missed by the model.

            <br><br>

            <b>Purple:</b>
            tumor correctly identified by the model
            and present in the ground truth.

            <br><br>

            <b>Red:</b>
            tumor predicted by the model but absent
            from the ground-truth mask.

            <br><br>

            A high Dice score should therefore
            correspond to a predominantly purple
            surface with relatively small blue and
            red regions.

            <br><br>

            The visualization does not modify
            the underlying segmentation results.

        </div>


    </div>

</div>


<!-- ========================================================
     PLOTLY
     ======================================================== -->

<script>
{plotly_js}
</script>


<!-- ========================================================
     APPLICATION
     ======================================================== -->

<script>


const patientData = {patient_json};


/* ==========================================================
   ELEMENTS
   ========================================================== */

const datasetSelect =
    document.getElementById(
        "datasetSelect"
    );

const patientSelect =
    document.getElementById(
        "patientSelect"
    );

const showGT =
    document.getElementById(
        "showGT"
    );

const showOverlap =
    document.getElementById(
        "showOverlap"
    );

const showPred =
    document.getElementById(
        "showPred"
    );


/* ==========================================================
   FILTER
   ========================================================== */

function getFilteredPatients() {{

    const dataset =
        datasetSelect.value;


    if (dataset === "all") {{

        return patientData;

    }}


    return patientData.filter(
        patient =>
            patient.split === dataset
    );

}}


/* ==========================================================
   POPULATE DROPDOWN
   ========================================================== */

function populatePatients(
    preferredPatient = null
) {{

    const patients =
        getFilteredPatients();


    patientSelect.innerHTML =
        "";


    patients.forEach(
        (patient, index) => {{

            const option =
                document.createElement(
                    "option"
                );


            option.value =
                index;


            option.textContent =
                patient.pid;


            patientSelect.appendChild(
                option
            );

        }}
    );


    if (patients.length === 0) {{

        return;

    }}


    let selectedIndex = 0;


    if (preferredPatient) {{

        const found =
            patients.findIndex(
                patient =>
                    patient.pid ===
                    preferredPatient
            );


        if (found >= 0) {{

            selectedIndex =
                found;

        }}

    }}


    patientSelect.value =
        selectedIndex;


    renderPatient(
        selectedIndex
    );

}}


/* ==========================================================
   FORMATTING
   ========================================================== */

function metric(value) {{

    return Number(
        value
    ).toFixed(4);

}}


function percent(value) {{

    return (
        Number(value)
        .toFixed(1)
        + "%"
    );

}}


function volume(value) {{

    return (
        Number(value)
        .toFixed(2)
        + " cm³"
    );

}}


function number(value) {{

    return Number(
        value
    ).toLocaleString();

}}


/* ==========================================================
   SIDEBAR
   ========================================================== */

function updateSidebar(
    patient
){{


    document.getElementById(
        "patientName"
    ).textContent =
        patient.pid;


    document.getElementById(
        "patientBadge"
    ).textContent =
        patient.split.toUpperCase();


    document.getElementById(
        "pcr"
    ).textContent =

        patient.pcr === null
        ? "N/A"
        : patient.pcr === 1
            ? "Yes"
            : "No";


    document.getElementById(
        "dice"
    ).textContent =
        metric(patient.dice);


    document.getElementById(
        "iou"
    ).textContent =
        metric(patient.iou);


    document.getElementById(
        "precision"
    ).textContent =
        metric(patient.precision);


    document.getElementById(
        "recall"
    ).textContent =
        metric(patient.recall);


    document.getElementById(
        "specificity"
    ).textContent =
        metric(patient.specificity);


    document.getElementById(
        "accuracy"
    ).textContent =
        metric(patient.accuracy);


    document.getElementById(
        "tp"
    ).textContent =
        number(patient.tp);


    document.getElementById(
        "fp"
    ).textContent =
        number(patient.fp);


    document.getElementById(
        "fn"
    ).textContent =
        number(patient.fn);


    document.getElementById(
        "gtVolume"
    ).textContent =
        volume(patient.gt_volume_cm3);


    document.getElementById(
        "predVolume"
    ).textContent =
        volume(patient.pred_volume_cm3);


    const diff =
        patient.volume_difference_cm3;


    document.getElementById(
        "volumeDifference"
    ).textContent =

        (
            diff >= 0
            ? "+"
            : ""
        )
        +
        Number(diff).toFixed(2)
        +
        " cm³ ("
        +
        (
            patient.volume_difference_percent
            >= 0
            ? "+"
            : ""
        )
        +
        Number(
            patient.volume_difference_percent
        ).toFixed(1)
        +
        "%)";


    document.getElementById(
        "overlapVolume"
    ).textContent =
        volume(
            patient.overlap_volume_cm3
        );


    document.getElementById(
        "overlapGT"
    ).textContent =
        percent(
            patient.overlap_percentage_gt
        );


    document.getElementById(
        "overlapPred"
    ).textContent =
        percent(
            patient.overlap_percentage_pred
        );


    document.getElementById(
        "split"
    ).textContent =
        patient.split;


    document.getElementById(
        "imageShape"
    ).textContent =
        patient.image_shape.join(
            " × "
        );


    document.getElementById(
        "voxelSpacing"
    ).textContent =
        patient.voxel_spacing
            .map(
                x =>
                    Number(x)
                    .toFixed(1)
            )
            .join(
                " × "
            )
        +
        " mm";

}}


/* ==========================================================
   MESH CREATION
   ========================================================== */

function makeMesh(
    meshData,
    name,
    color,
    opacity
){{


    if (
        !meshData ||
        !meshData.vertices ||
        meshData.vertices.length === 0
    ){{

        return null;

    }}


    const vertices =
        meshData.vertices;


    const faces =
        meshData.faces;


    return {{

        type:
            "mesh3d",


        x:
            vertices.map(
                v => v[0]
            ),


        y:
            vertices.map(
                v => v[1]
            ),


        z:
            vertices.map(
                v => v[2]
            ),


        i:
            faces.map(
                f => f[0]
            ),


        j:
            faces.map(
                f => f[1]
            ),


        k:
            faces.map(
                f => f[2]
            ),


        name:
            name,


        color:
            color,


        opacity:
            opacity,


        flatshading:
            false,


        lighting: {{

            ambient:
                0.55,

            diffuse:
                0.85,

            specular:
                0.45,

            roughness:
                0.32,

            fresnel:
                0.18

        }},


        lightposition: {{

            x:
                100,

            y:
                200,

            z:
                300

        }},


        hoverinfo:
            "skip"

    }};

}}


/* ==========================================================
   RENDER PATIENT
   ========================================================== */

function renderPatient(
    index
){{


    const patients =
        getFilteredPatients();


    const patient =
        patients[index];


    if (!patient) {{

        return;

    }}


    updateSidebar(
        patient
    );


    const traces = [];


    /* ------------------------------------------------------
       GT ONLY
       ------------------------------------------------------ */

    if (showGT.checked) {{

        const mesh =
            makeMesh(
                patient.gt_only,
                "Ground Truth only",
                "#3b82f6",
                0.50
            );


        if (mesh) {{

            traces.push(
                mesh
            );

        }}

    }}


    /* ------------------------------------------------------
       OVERLAP
       ------------------------------------------------------ */

    if (showOverlap.checked) {{

        const mesh =
            makeMesh(
                patient.overlap,
                "Correct overlap",
                "#a855f7",
                0.50
            );


        if (mesh) {{

            traces.push(
                mesh
            );

        }}

    }}


    /* ------------------------------------------------------
       PREDICTION ONLY
       ------------------------------------------------------ */

    if (showPred.checked) {{

        const mesh =
            makeMesh(
                patient.pred_only,
                "Prediction only",
                "#ef4444",
                0.50
            );


        if (mesh) {{

            traces.push(
                mesh
            );

        }}

    }}


    /* ------------------------------------------------------
       LAYOUT
       ------------------------------------------------------ */

    const layout = {{

        paper_bgcolor:
            "#080c11",


        plot_bgcolor:
            "#080c11",


        font: {{

            color:
                "#d8e1ea"

        }},


        margin: {{

            l:
                0,

            r:
                0,

            t:
                0,

            b:
                0

        }},


        scene: {{

            bgcolor:
                "#080c11",


            aspectmode:
                "data",


            xaxis: {{

                visible:
                    false

            }},


            yaxis: {{

                visible:
                    false

            }},


            zaxis: {{

                visible:
                    false

            }},


            camera: {{

                eye: {{

                    x:
                        1.6,

                    y:
                        1.6,

                    z:
                        1.25

                }}

            }}


        }},


        showlegend:
            false

    }};


    const config = {{

        responsive:
            true,


        displaylogo:
            false,


        scrollZoom:
            true,


        modeBarButtonsToAdd:
            [
                "resetCameraDefault"
            ]

    }};


    Plotly.react(
        "viewer",
        traces,
        layout,
        config
    );

}}


/* ==========================================================
   EVENTS
   ========================================================== */

datasetSelect.addEventListener(
    "change",
    () => {{

        populatePatients();

    }}
);


patientSelect.addEventListener(
    "change",
    () => {{

        renderPatient(
            Number(
                patientSelect.value
            )
        );

    }}
);


showGT.addEventListener(
    "change",
    () => {{

        renderPatient(
            Number(
                patientSelect.value
            )
        );

    }}
);


showOverlap.addEventListener(
    "change",
    () => {{

        renderPatient(
            Number(
                patientSelect.value
            )
        );

    }}
);


showPred.addEventListener(
    "change",
    () => {{

        renderPatient(
            Number(
                patientSelect.value
            )
        );

    }}
);


/* ==========================================================
   INITIALIZE
   ========================================================== */

populatePatients();

</script>


</body>

</html>
"""


# ============================================================
# SAVE
# ============================================================

OUTPUT_HTML.write_text(
    html,
    encoding="utf-8"
)


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("DETAILED 3D VIEWER CREATED")
print("=" * 70)

print(
    "\nPatients:",
    len(all_patients)
)

print(
    "Validation:",
    sum(
        p["split"] == "validation"
        for p in all_patients
    )
)

print(
    "Test:",
    sum(
        p["split"] == "test"
        for p in all_patients
    )
)

print(
    "\nThreshold:",
    THRESHOLD
)

print(
    "\nSaved to:"
)

print(
    OUTPUT_HTML
)

print(
    "\nFile size:",
    f"{OUTPUT_HTML.stat().st_size/(1024**2):.2f} MB"
)

print("\n✓ Empty-mask handling")
print("✓ GT-only visualization")
print("✓ Overlap visualization")
print("✓ Prediction-only visualization")
print("✓ Dice / IoU / Precision / Recall")
print("✓ Specificity / Accuracy")
print("✓ TP / FP / FN")
print("✓ Tumor volume in cm³")
print("✓ Volume difference")
print("✓ Spatial overlap")
print("✓ Validation/Test filtering")
print("✓ Patient selection")
print("✓ Interactive 3D rotation")
print("✓ Zoom / pan")
print("✓ 3D lighting")
print("✓ Voxel spacing")
print("✓ Image dimensions")

print(
    "\n" + "=" * 70
)

In [ ]:
from google.colab import files

files.download(str(OUTPUT_HTML))